### **Dataset Handling**

In [ ]:
from typing import Any, Callable
from pathlib import Path
from PIL import Image

from tqdm import tqdm
import torch
import torch.nn as nn
from torch.types import Tensor
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as ts
from torchvision.transforms import Compose

import spark as pk

In [3]:
import sys

def get_size(obj: object, default: Any = -1) -> int:
    """Computes how much memory storage is being used by input obj in [bytes]."""
    return sys.getsizeof(obj, default=default)

In [ ]:
from spark.handle import get_data_filespaths
from spark.processing import center_tensor, normalise


class Normalise(nn.Module):
    def __init__(self, norm_range: str = 'unilateral') -> None:
        super().__init__()
        self.norm_range = norm_range
        return
    
    def __call__(self, tensor: Tensor):
        return normalise(tensor, self.norm_range)
    

class OutputManager:
    """
    Manager for a model output data object. This class acts as a manager for data,
    and when called stores module output tensor, detaching it from the operations
    graph and moving it to the CPU to avoid GPU memory RAM overload.
    NOTE:
        * As of now, only Tensor data type handling is supported as module output.
    """
    def __init__(self, storage: list[Tensor] | None = None) -> None:
        self.storage = storage if storage is not None else []
    
    def __call__(self, module: nn.Module, in_data: Any, out_data: Tensor) -> None:
        # # NOTE: manage PyTorch multi-tensor outputs
        # if isinstance(out_data, tuple):
        #     out_data = out_data[0]
        
        if not isinstance(out_data, Tensor):
            raise ValueError(
                f"Invalid 'out_data' type {type(out_data)}, must be 'torch.Tensor'."
            )
        self.storage.append(out_data.detach().cpu())
    
    def clear(self) -> None:
        """Clears the whole storage content."""
        self.storage = []
    
    def merge(self) -> Tensor:
        """Merges the storage content to single `torch.Tensor`."""
        if not self.storage:
            return torch.empty(0)
        return torch.cat(self.storage, dim=0)


class ImageDataset(Dataset):
    """
    Baseline AutoEncoder dataset configurator.
    """
    def __init__(self, data_path: str | Path, transform: Compose | None) -> None:
        files_list = get_data_filespaths(data_path, shuffle=True)
        self.data = sp.process_data(files_list, lambda img: Image.open(img).convert('L'), transform)
    
    def __len__(self) -> int:
        return len(self.data)
    
    def __getitem__(self, index) -> tuple[Tensor, Tensor]:
        sample = self.data[index]
        return sample, sample

In [5]:
#BASEPATH: str = '/home/edoardo/Desktop/MockDataForDMs'
BASEPATH: str = '/mnt/d/MockDataForDMs'
DATASET_ID: str = 'mockPolyImgsDataset'
BATCH_SIZE: int = 250
VALID_SIZE: float = 0.2
PREPROCESSING: Compose = ts.Compose(
    [
        ts.PILToTensor(),
        center_tensor,
        normalise,
    ]
)

In [6]:
def handle_dataset(*filepaths: str | Path, **kwargs) -> tuple[DataLoader, DataLoader | None]:
    """Handles dataset(s). A dataset is loaded if its file exists, else is generated and saved."""
    raise NotImplementedError


try:
    dataset: Dataset = sp.load_dataset(f'{BASEPATH}/{DATASET_ID}.pt')
except FileNotFoundError:
    print('Dataset not found, generating...')
    dataset: ImageDataset = ImageDataset(f'{BASEPATH}/ImgsMockDatasetDMs', PREPROCESSING)
    sp.save_dataset(dataset, f'{BASEPATH}/{DATASET_ID}.pt')

train_dl, valid_dl = sp.get_dataloaders(dataset, BATCH_SIZE, VALID_SIZE)

Loading dataset...
Dataset loaded!
Baking DataLoaders...
DataLoaders ready-to-go!


### **Model Handling**

In [7]:
def save_checkpoint() -> None:
    """Saves a checkpoint during training, given a metric."""
    raise NotImplementedError

In [8]:
from basic_mockModels import MockModel
from fn_repo import test_model_works

MODEL_ID: str = 'mockCNNbasicModel'
DATA_SHAPE: torch.Size = torch.Size([1, 84, 84])
OUT_FEATURES: int = 4

# try:
#     model_data: dict[str, Any] = sp.load_model(f'{BASEPATH}/{MODEL_ID}.pt')
#     model: nn.Module = model_data['state_dict']
# except FileNotFoundError:
#     print('Model not found, generating...')
#     model: nn.Module = MockModel(data_shape=DATA_SHAPE, out_features=OUT_FEATURES)
#     sp.save_model(model, f'{BASEPATH}/{MODEL_ID}.pt')

model: nn.Module = MockModel(data_shape=DATA_SHAPE, out_features=OUT_FEATURES)
print(model)
# out = test_model_works(model, DATA_SHAPE)

MockModel(
  (net): Sequential(
    (0): Conv2d(1, 16, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(16, 16, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU()
    (6): Conv2d(16, 1, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): BatchNorm2d(1, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (8): ReLU()
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Flatten(start_dim=1, end_dim=-1)
    (11): Linear(in_features=1764, out_features=1024, bias=True)
    (12): ReLU()
    (13): Dropout(p=0.3, inplace=False)
    (14): Linear(in_features=1024, out_features=4, bias=True)
    (15): Softmax(dim=1)
  )
)


### **AutoEncoder Model Handling**

In [9]:
from basic_mockModels import MockAutoEncoder

IN_CHANNELS: int = 1
LATENT_DIM: int = 8

autoencoder = MockAutoEncoder(IN_CHANNELS, LATENT_DIM)
print(autoencoder)
# out = test_model_works(autoencoder, DATA_SHAPE)  ---- WORKS!

MockAutoEncoder(
  (encoder): Encoder(
    (architecture): Sequential(
      (0): Conv2d(1, 32, kernel_size=(5, 5), stride=(1, 1))
      (1): ReLU()
      (2): Conv2d(32, 32, kernel_size=(5, 5), stride=(1, 1))
      (3): ReLU()
      (4): Conv2d(32, 32, kernel_size=(4, 4), stride=(2, 2))
      (5): ReLU()
      (6): Conv2d(32, 32, kernel_size=(3, 3), stride=(2, 2))
      (7): ReLU()
      (8): Conv2d(32, 8, kernel_size=(4, 4), stride=(1, 1))
    )
  )
  (decoder): Decoder(
    (architecture): Sequential(
      (0): ConvTranspose2d(8, 32, kernel_size=(4, 4), stride=(1, 1))
      (1): ReLU()
      (2): ConvTranspose2d(32, 32, kernel_size=(3, 3), stride=(2, 2))
      (3): ReLU()
      (4): ConvTranspose2d(32, 32, kernel_size=(4, 4), stride=(2, 2))
      (5): ReLU()
      (6): ConvTranspose2d(32, 32, kernel_size=(5, 5), stride=(1, 1))
      (7): ReLU()
      (8): ConvTranspose2d(32, 1, kernel_size=(5, 5), stride=(1, 1))
    )
  )
)


In [ ]:
import itertools as it
from typing import NamedTuple
import warnings

import torch.optim as opt
from torch.amp import GradScaler, autocast

from spark.inspect import forward_data_capture


class TrainResults(NamedTuple):
    """
    Model trainer results.
    NOTE:
        * the tensors in `latent_dmap` are still attached to the Op. Graph
    """
    train_loss: list[float]
    valid_loss: list[float]
    latent_dmap: dict[int, Tensor]


def train_model(
    params: dict[str, Any],
    train_dl: DataLoader,
    epochs: int,
    learning_rate: float,
    latent_space_hook: OutputManager,
    valid_dl: DataLoader | None = None,
) -> TrainResults:
    """
    Basic train routine.
    """
    def check_loss_val(loss_val: float, msg: str) -> bool:
        """Checks the current loss value and raises a warning if is NaN."""
        if not torch.isnan(loss_val):
            return True
        warnings.warn(msg)
        return False

    # config procedure/loss container
    nbatches: int = len(train_dl)
    total_steps: int = epochs * nbatches
    avg_train_loss, avg_valid_loss = [], []
    latent_container: dict[int, Tensor] = {}

    # setup model/optimiser/scaler for memory saving
    device = params['device']
    model = params['model'].to(device)
    loss_fn = params['loss']
    optimiser = params['optimiser'](model.parameters(), lr=learning_rate)
    scheduler = params['scheduler'](optimiser, patience=5, factor=0.5)
    scaler = GradScaler(device)

    # train model
    train_iter = tqdm(
        iterable=enumerate(it.islice(it.cycle(train_dl), total_steps)),
        desc='Training Model',
    )
    running_batches = 0
    running_train_loss = 0.0

    model.train()
    for idx, (input_batch, trg_batch) in train_iter:
        # update tqdm bar to keep track of the routine
        epoch, batch = divmod(idx, nbatches)
        current_lr = optimiser.param_groups[0]['lr']
        info: dict = {
            'epoch': f'{epoch + 1}/{epochs}',
            'batch': f'{batch + 1}/{nbatches}',
            'lr': f'{current_lr:.2e}'
        }
        train_iter.set_postfix(info)

        # model optimisation
        input_batch, trg_batch = input_batch.to(device), trg_batch.to(device)

        optimiser.zero_grad()
        with autocast(device_type=device):
            out = model(input_batch)
            loss_val = loss_fn(out, trg_batch)
        
        if check_loss_val(
            loss_val, f'Train loss NaN @ E: {epoch + 1} B: {batch + 1}',
        ):
            running_batches += 1
            running_train_loss += loss_val.item()

            scaler.scale(loss_val).backward()
            scaler.step(optimiser)
            scaler.update()

        # compute avg loss val at the end of the epoch
        if (idx + 1) % nbatches == 0:
            avg_train_loss.append(running_train_loss / max(running_batches, 1))
            running_batches = 0
            running_train_loss = 0.0
        
            # ------------- validation step -------------
            if valid_dl is not None:
                model.eval()
                running_valid_batches = 0
                running_valid_loss = 0.0

                with torch.no_grad(), forward_data_capture(model.encoder, latent_space_hook):
                    for v_input, v_trg in valid_dl:
                        v_input, v_trg = v_input.to(device), v_trg.to(device)

                        with autocast(device_type=device):
                            v_out = model(v_input)
                            v_loss_val = loss_fn(v_out, v_trg)
                        
                        if check_loss_val(
                            v_loss_val, f'Valid loss NaN @ E: {epoch + 1}',
                        ):
                            running_valid_batches += 1
                            running_valid_loss += v_loss_val.item()
                    
                # store avg valid loss
                avg_v_loss_val = running_valid_loss / max(running_valid_batches, 1)
                avg_valid_loss.append(avg_v_loss_val)
                # update lr val through scheduler
                scheduler.step(avg_v_loss_val)
                # merge latent space vectors for current epoch and clear storage
                latent_container[epoch] = latent_space_hook.merge()
                latent_space_hook.clear()

                model.train()

    results = TrainResults(avg_train_loss, avg_valid_loss, latent_container)

    return results

In [11]:
from basic_train_routine import training_setup

EPOCHS: int = 5
LR: int = 1e-2
DEVICE: int = 'cuda' if torch.cuda.is_available() else 'cpu'

# ae_loss = lambda x, y: nn.MSELoss()(x, y) - nn.KLDivLoss()(x, y)
ae_loss = nn.MSELoss()

params = training_setup(
    autoencoder, ae_loss, opt.Adam, opt.lr_scheduler.ReduceLROnPlateau, DEVICE,
)
latent_space_hook = OutputManager()

avg_train_loss, avg_valid_loss, latent_container = train_model(
    params=params,
    train_dl=train_dl,
    epochs=EPOCHS,
    learning_rate=LR,
    latent_space_hook=latent_space_hook,
    valid_dl=valid_dl,
)

## Operating on device: cuda.
## Model parameters: 112457.


Training Model: 65it [00:07,  8.21it/s, epoch=5/5, batch=13/13, lr=1.00e-02]


In [12]:
avg_train_loss, avg_valid_loss

([0.7894843782369907,
  0.15310843116961992,
  0.14306694383804613,
  0.13463976406134093,
  0.1340040106039781],
 [0.13168494030833244,
  0.14619702473282814,
  0.13856033235788345,
  0.1351836435496807,
  0.13414587825536728])

In [13]:
s = latent_container[0].shape

s, get_size(s), get_size(latent_container)

(torch.Size([800, 8, 15, 15]), 80, 224)

In [14]:
len(latent_container)

5